# Can Saliency Maps Flag Chest Radiograph Classifier Errors at a New Hospital? An External Validation Study
## External Data Preparation
## Aashish Chapagai (Last Edited: October 2026)

In [ ]:
import os
import re
import glob
import json
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pydicom
from PIL import Image
from pycocotools import mask as coco_mask
try:
    from pydicom.pixels import apply_modality_lut, apply_voi_lut
except ImportError:
    from pydicom.pixel_data_handlers.util import apply_modality_lut, apply_voi_lut

os.makedirs('models', exist_ok=True)
os.makedirs('figures', exist_ok=True)

In [ ]:
EXTERNAL_DIR = 'external'
CHEXLOCALIZE_DIR = os.path.join(EXTERNAL_DIR, 'chexlocalize')
CHEXMASK_DIR = os.path.join(EXTERNAL_DIR, 'chexmask')
VINDR_DIR = os.path.join(EXTERNAL_DIR, 'vindr')
NIH_SIZE = 1024
MASK_QUALITY_MIN = 0.7
REGION_DILATION = 0.025
BASE_MARGIN = 0.04
CHUNK_ROWS = 1000
IMAGE_EXTENSIONS = ('.jpg', '.jpeg', '.png')
DICOM_EXTENSIONS = ('.dicom', '.dcm')
RESAMPLE = getattr(Image, 'Resampling', Image).BICUBIC
IMG_SIZE = np.load('models/X_test.npy', mmap_mode='r').shape[1]
print(f"Preparing external data at {IMG_SIZE}px.")

In [ ]:
CHEXPERT_COLUMNS = {
    'Atelectasis': 'Atelectasis',
    'Cardiomegaly': 'Cardiomegaly',
    'Consolidation': 'Consolidation',
    'Edema': 'Edema',
    'Effusion': 'Pleural Effusion',
    'Lesion': 'Lung Lesion',
    'Pneumonia': 'Pneumonia',
    'Pneumothorax': 'Pneumothorax',
}

CHEXLOCALIZE_KEYS = {finding: column for finding, column in CHEXPERT_COLUMNS.items() if finding != 'Pneumonia'}

VINDR_COLUMNS = {
    'Atelectasis': 'atelectasis',
    'Cardiomegaly': 'cardiomegaly',
    'Consolidation': 'consolidation',
    'Edema': 'edema',
    'Effusion': 'pleural effusion',
    'Emphysema': 'emphysema',
    'Fibrosis': 'pulmonary fibrosis',
    'Infiltration': 'infiltration',
    'Lesion': 'nodule/mass',
    'Pleural thickening': 'pleural thickening',
    'Pneumonia': 'pneumonia',
    'Pneumothorax': 'pneumothorax',
}

NIH_BOX_FINDINGS = {
    'Atelectasis': 'Atelectasis',
    'Cardiomegaly': 'Cardiomegaly',
    'Effusion': 'Effusion',
    'Infiltrate': 'Infiltration',
    'Infiltration': 'Infiltration',
    'Mass': 'Lesion',
    'Nodule': 'Lesion',
    'Pneumonia': 'Pneumonia',
    'Pneumothorax': 'Pneumothorax',
}

CHEXPERT_ID = re.compile(r'(patient\d+)[/_](study\d+)[/_](view\d+_(?:frontal|lateral))', re.IGNORECASE)

In [ ]:
def find_file(root, name):
    hits = sorted(glob.glob(os.path.join(root, '**', name), recursive=True))
    if not hits:
        print(f"Could not find {name} anywhere under {root}.")
        return None
    return hits[0]

def chexpert_key(path):
    match = CHEXPERT_ID.search(str(path).replace('\\', '/'))
    return '_'.join(part.lower() for part in match.groups()) if match else None

def load_image(path):
    with Image.open(path) as handle:
        image = handle.convert('L')
        size = image.size
        image = image.resize((IMG_SIZE, IMG_SIZE), RESAMPLE)
    return np.asarray(image, dtype=np.uint8)[..., None], size

def read_dicom(path):
    dataset = pydicom.dcmread(path)
    pixels = apply_modality_lut(dataset.pixel_array, dataset).round().astype(np.int32)
    pixels = apply_voi_lut(pixels, dataset).astype(np.float32)
    if str(dataset.get('PhotometricInterpretation', '')).upper() == 'MONOCHROME1':
        pixels = pixels.max() - pixels
    pixels -= pixels.min()
    return np.uint8(np.round(255 * pixels / max(float(pixels.max()), 1e-6)))

def image_id(path):
    return os.path.splitext(os.path.basename(str(path)))[0]

In [ ]:
def shrink(mask):
    resized = cv2.resize(mask.astype(np.float32), (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    return resized >= 0.5

def decode_rle(rle, height, width):
    counts = np.zeros(height * width + 1, dtype=np.int32)
    if isinstance(rle, str) and rle.strip():
        runs = np.asarray(rle.split(), dtype=np.int64)
        np.add.at(counts, runs[0::2] - 1, 1)
        np.add.at(counts, runs[0::2] - 1 + runs[1::2], -1)
    return (np.cumsum(counts[:-1]) > 0).reshape(height, width)

def decode_coco(entry):
    counts = entry['counts']
    rle = {'size': [int(v) for v in entry['size']], 'counts': counts.encode() if isinstance(counts, str) else counts}
    return coco_mask.decode(rle).astype(bool)

In [ ]:
def anatomy_layers(left, right, heart):
    lungs = cv2.resize((left | right).astype(np.uint8), (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
    organs = cv2.resize((left | right | heart).astype(np.uint8), (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
    region = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.uint8)
    points = cv2.findNonZero(organs)
    if points is not None:
        points = points.reshape(-1, 2)
        lung_y, lung_x = np.nonzero(lungs)
        if len(lung_x):
            floor = min(IMG_SIZE - 1, int(lung_y.max() + round(BASE_MARGIN * IMG_SIZE)))
            points = np.vstack([points, [[lung_x.min(), floor], [lung_x.max(), floor]]])
        cv2.fillConvexPoly(region, cv2.convexHull(points.astype(np.int32)), 1)
        radius = max(1, int(round(REGION_DILATION * IMG_SIZE)))
        region = cv2.dilate(region, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * radius + 1, 2 * radius + 1)))
    return region | (lungs << 1)

def lung_offsets(left, right):
    left_y, left_x = np.nonzero(left)
    right_y, right_x = np.nonzero(right)
    if len(left_x) == 0 or len(right_x) == 0:
        return np.nan, np.nan
    return abs(left_x.mean() - right_x.mean()), abs(left_y.mean() - right_y.mean())

In [ ]:
def build_anatomy(csv_path, keys, normalise, output_path, quality_column='Dice RCA (Mean)', default_size=None):
    position = {key: i for i, key in enumerate(keys)}
    wanted = set(position)
    header = pd.read_csv(csv_path, nrows=0).columns.tolist()
    lookup = {name.strip().lower(): name for name in header}
    required = [quality_column, 'Left Lung', 'Right Lung', 'Heart']
    fields = {field: lookup.get(field.lower()) for field in required + ['Height', 'Width']}
    missing = [field for field in required if fields[field] is None]
    csv_name = os.path.basename(csv_path)
    sized = fields['Height'] is not None and fields['Width'] is not None
    print(f"{csv_name}: required columns present: {not missing}")
    print(f"{csv_name}: mask size columns present: {sized}")
    anatomy = np.lib.format.open_memmap(output_path, mode='w+', dtype=np.uint8, shape=(len(keys), IMG_SIZE, IMG_SIZE))
    quality = np.full(len(keys), np.nan, dtype=np.float32)
    shapes = np.full((len(keys), 2), np.nan, dtype=np.float32)
    offsets = []
    columns = [header[0]] + [name for name in fields.values() if name is not None]
    for chunk in pd.read_csv(csv_path, usecols=columns, chunksize=CHUNK_ROWS):
        chunk_keys = chunk[header[0]].astype(str).map(normalise)
        matched = chunk_keys.isin(wanted).to_numpy()
        for key, record in zip(chunk_keys[matched], chunk[matched].to_dict('records')):
            height = int(record[fields['Height']]) if fields['Height'] else default_size
            width = int(record[fields['Width']]) if fields['Width'] else default_size
            left = decode_rle(record[fields['Left Lung']], height, width)
            right = decode_rle(record[fields['Right Lung']], height, width)
            heart = decode_rle(record[fields['Heart']], height, width)
            row = position[key]
            anatomy[row] = anatomy_layers(left, right, heart)
            quality[row] = float(record[fields[quality_column]])
            shapes[row] = (height, width)
            if len(offsets) < 500:
                offsets.append(lung_offsets(left, right))
    anatomy.flush()
    del anatomy
    offsets = np.asarray(offsets, dtype=np.float64).reshape(-1, 2)
    horizontal = float(np.mean(offsets[:, 0] > offsets[:, 1])) if len(offsets) else float('nan')
    return quality, shapes, horizontal

def report_anatomy(name, quality, horizontal, check=f'RCA Dice >= {MASK_QUALITY_MIN}'):
    found = int(np.isfinite(quality).sum())
    usable = int((quality >= MASK_QUALITY_MIN).sum())
    print(f"{name}: anatomy masks found for {found} of {len(quality)} images.")
    print(f"{name}: {usable} masks pass the {check} check.")
    print(f"{name}: left and right lungs sit side by side in {100 * horizontal:.1f}% of sampled masks.")
    print(f"{name}: lung masks look transposed: {bool(np.isfinite(horizontal) and horizontal < 0.9)}")

In [ ]:
nih_metadata = pd.read_csv('models/metadata_test.csv')
nih_keys = nih_metadata['Image Index'].astype(str).map(os.path.basename).tolist()
print(f"Decoding CheXmask anatomy for {len(nih_keys)} NIH test images.")
nih_csv = os.path.join(CHEXMASK_DIR, 'ChestX-Ray8.csv')
nih_path = 'models/anatomy_nih.npy'
nih_ready = os.path.exists(nih_csv)
print(f"CheXmask NIH masks found: {nih_ready}")
if nih_ready:
    nih_quality, nih_shapes, nih_horizontal = build_anatomy(nih_csv, nih_keys, os.path.basename, nih_path, default_size=NIH_SIZE)
    np.save('models/anatomy_nih_quality.npy', nih_quality)
    report_anatomy('NIH', nih_quality, nih_horizontal)

In [ ]:
bbox = pd.read_csv('BBox.csv').iloc[:, :6]
bbox.columns = ['image', 'label', 'x', 'y', 'w', 'h']
nih_row_of = {key: i for i, key in enumerate(nih_keys)}
bbox = bbox[bbox['image'].isin(set(nih_row_of)) & bbox['label'].isin(set(NIH_BOX_FINDINGS))]
scale = IMG_SIZE / NIH_SIZE
nih_boxes = pd.DataFrame({
    'row': bbox['image'].map(nih_row_of).astype(int),
    'finding': bbox['label'].map(NIH_BOX_FINDINGS),
    'x0': bbox['x'] * scale,
    'y0': bbox['y'] * scale,
    'x1': (bbox['x'] + bbox['w']) * scale,
    'y1': (bbox['y'] + bbox['h']) * scale,
})
nih_boxes.to_csv('models/gt_boxes_nih.csv', index=False)
print(f"NIH: {len(nih_boxes)} radiologist boxes on {nih_boxes['row'].nunique()} test images.")
print(nih_boxes['finding'].value_counts().to_string())

In [ ]:
label_paths = {split: find_file(CHEXLOCALIZE_DIR, f'{split}_labels.csv') for split in ('val', 'test')}
image_paths = {}
for path in sorted(glob.glob(os.path.join(CHEXLOCALIZE_DIR, '**', '*'), recursive=True)):
    key = chexpert_key(path) if os.path.splitext(path)[1].lower() in IMAGE_EXTENSIONS else None
    if key is not None:
        image_paths.setdefault(key, path)
records = []
unmatched = 0
for split, label_path in label_paths.items():
    table = pd.read_csv(label_path)
    missing_labels = [column for column in CHEXPERT_COLUMNS.values() if column not in table.columns]
    print(f"{os.path.basename(label_path)}: missing label columns: {missing_labels or 'none'}")
    path_column = 'Path' if 'Path' in table.columns else table.columns[0]
    for row in table.to_dict('records'):
        key = chexpert_key(row[path_column])
        if key is None or not key.endswith('frontal'):
            continue
        if key not in image_paths:
            unmatched += 1
            continue
        view = row.get('AP/PA', '')
        record = {
            'key': key,
            'split': split,
            'patient': key.split('_')[0],
            'path': image_paths[key],
            'view': view if isinstance(view, str) else '',
        }
        for finding, column in CHEXPERT_COLUMNS.items():
            record[finding] = int(float(row[column]) >= 0.5) if pd.notna(row[column]) else 0
        records.append(record)
chexpert = pd.DataFrame(records).drop_duplicates('key').reset_index(drop=True)
print(f"CheXpert: {len(chexpert)} frontal radiographs from {chexpert['patient'].nunique()} patients.")
print(f"Validation split: {int((chexpert['split'] == 'val').sum())} images.")
print(f"Test split: {int((chexpert['split'] == 'test').sum())} images.")
print(f"Labelled frontal images without a file on disk: {unmatched}.")
print(chexpert[list(CHEXPERT_COLUMNS)].sum().to_string())

In [ ]:
X_chexpert = np.zeros((len(chexpert), IMG_SIZE, IMG_SIZE, 1), dtype=np.uint8)
sizes = np.zeros((len(chexpert), 2), dtype=np.int64)
for i, path in enumerate(chexpert['path']):
    X_chexpert[i], sizes[i] = load_image(path)
chexpert['width'] = sizes[:, 0]
chexpert['height'] = sizes[:, 1]
np.save('models/X_chexpert.npy', X_chexpert)
np.save('models/Y_chexpert.npy', chexpert[list(CHEXPERT_COLUMNS)].to_numpy(dtype=np.uint8))
np.save('models/chexpert_findings.npy', np.array(list(CHEXPERT_COLUMNS)))
chexpert.to_csv('models/metadata_chexpert.csv', index=False)
print(f"Saved {len(chexpert)} CheXpert images at {IMG_SIZE}px.")

In [ ]:
def collect_masks(json_path, keyed_rows, names):
    with open(json_path) as handle:
        segmentations = json.load(handle)
    rows, findings, masks = [], [], []
    for row, key in keyed_rows:
        entry = segmentations.get(key, {})
        for finding, name in names.items():
            if name in entry:
                mask = decode_coco(entry[name])
                if mask.any():
                    rows.append(row)
                    findings.append(finding)
                    masks.append(shrink(mask))
    stacked = np.stack(masks) if masks else np.zeros((0, IMG_SIZE, IMG_SIZE), dtype=bool)
    return np.asarray(rows, dtype=np.int64), np.asarray(findings, dtype=str), stacked

def first_point(value):
    if isinstance(value, dict):
        if 'x' in value and 'y' in value:
            return float(value['x']), float(value['y'])
        value = list(value.values())
    if isinstance(value, (list, tuple)):
        if len(value) >= 2 and all(isinstance(v, (int, float)) for v in value[:2]):
            return float(value[0]), float(value[1])
        for item in value:
            point = first_point(item)
            if point is not None:
                return point
    return None

In [ ]:
keyed_rows = {}
for split in ('val', 'test'):
    rows = np.flatnonzero(chexpert['split'] == split)
    keyed_rows[split] = list(zip(rows, chexpert.loc[chexpert['split'] == split, 'key']))
parts = []
for split in ('val', 'test'):
    json_path = find_file(CHEXLOCALIZE_DIR, f'gt_segmentations_{split}.json')
    parts.append(collect_masks(json_path, keyed_rows[split], CHEXLOCALIZE_KEYS))
gt_rows = np.concatenate([part[0] for part in parts])
gt_findings = np.concatenate([part[1] for part in parts])
gt_masks = np.concatenate([part[2] for part in parts])
np.savez_compressed('models/gt_masks_chexpert.npz', rows=gt_rows, findings=gt_findings, masks=gt_masks)
print(f"CheXpert: {len(gt_rows)} radiologist segmentations on {len(np.unique(gt_rows))} images.")
print(pd.Series(gt_findings).value_counts().to_string())

In [ ]:
hb_json = find_file(CHEXLOCALIZE_DIR, 'hb_segmentations_test.json')
hb_rows, hb_findings, hb_masks = collect_masks(hb_json, keyed_rows['test'], CHEXLOCALIZE_KEYS)
np.savez_compressed('models/hb_masks_chexpert.npz', rows=hb_rows, findings=hb_findings, masks=hb_masks)
annotation_size = {}
for name in ('gt_segmentations_val.json', 'gt_segmentations_test.json', 'hb_segmentations_test.json'):
    with open(find_file(CHEXLOCALIZE_DIR, name)) as handle:
        entries = json.load(handle)
    for key, entry in entries.items():
        if entry:
            annotation_size[key] = tuple(int(v) for v in next(iter(entry.values()))['size'])
annotated = [(row, key) for row, key in enumerate(chexpert['key']) if key in annotation_size]
matching = 0
for row, key in annotated:
    if annotation_size[key] == (chexpert.at[row, 'height'], chexpert.at[row, 'width']):
        matching += 1
print(f"CheXpert: downloaded image size equals the annotated image size for {matching}/{len(annotated)} annotated images.")

In [ ]:
with open(find_file(CHEXLOCALIZE_DIR, 'hb_salient_pt_test.json')) as handle:
    salient = json.load(handle)
points = []
for row, key in keyed_rows['test']:
    entry = salient.get(key, {})
    image_shape = (chexpert.at[row, 'height'], chexpert.at[row, 'width'])
    height, width = entry.get('img_size', annotation_size.get(key, image_shape))
    for finding, name in CHEXLOCALIZE_KEYS.items():
        point = first_point(entry.get(name))
        if point is not None:
            points.append({
                'row': row,
                'finding': finding,
                'x': point[0] * IMG_SIZE / float(width),
                'y': point[1] * IMG_SIZE / float(height),
            })
pd.DataFrame(points, columns=['row', 'finding', 'x', 'y']).to_csv('models/hb_points_chexpert.csv', index=False)
print(f"CheXpert human benchmark: {len(hb_rows)} segmentations and {len(points)} most-representative points.")

In [ ]:
hybridgnet_csv = 'models/hybridgnet_masks_chexpert.csv'
chexpert_anatomy_ready = os.path.exists(hybridgnet_csv)
print(f"HybridGNet CheXpert masks found: {chexpert_anatomy_ready}")
if chexpert_anatomy_ready:
    chexpert_keys = chexpert['key'].tolist()
    cx_path = 'models/anatomy_chexpert.npy'
    cx_quality, cx_shapes, cx_horizontal = build_anatomy(hybridgnet_csv, chexpert_keys, chexpert_key, cx_path, 'Quality')
    np.save('models/anatomy_chexpert_quality.npy', cx_quality)
    report_anatomy('CheXpert', cx_quality, cx_horizontal, check='anatomical plausibility')
    for split in ('val', 'test'):
        rows = (chexpert['split'] == split).to_numpy()
        available = int(np.isfinite(cx_quality[rows]).sum())
        print(f"  Split {split}: anatomy available for {available} of {int(rows.sum())} images.")
    found = np.isfinite(cx_shapes[:, 0])
    mask_aspect = cx_shapes[found, 0] / cx_shapes[found, 1]
    image_aspect = chexpert['height'].to_numpy()[found] / chexpert['width'].to_numpy()[found]
    aspect_gap = np.abs(mask_aspect - image_aspect)
    agreeing = int((aspect_gap < 0.02).sum())
    print(f"  Mask and image aspect ratios agree within 2% for {agreeing} of {int(found.sum())} images.")

In [ ]:
vindr_ready = bool(glob.glob(os.path.join(VINDR_DIR, '**', 'image_labels_test.csv'), recursive=True))
if vindr_ready:
    labels = pd.read_csv(find_file(VINDR_DIR, 'image_labels_test.csv'))
    lookup = {column.strip().lower(): column for column in labels.columns}
    missing_columns = [name for name in VINDR_COLUMNS.values() if name not in lookup]
    print(f"VinDr-CXR label columns matched: {len(VINDR_COLUMNS) - len(missing_columns)} of {len(VINDR_COLUMNS)}.")
    print(f"Missing columns: {missing_columns or 'none'}")
    vindr_findings = [finding for finding, name in VINDR_COLUMNS.items() if name in lookup]
    id_column = lookup.get('image_id', labels.columns[0])
    dicoms = {}
    for path in sorted(glob.glob(os.path.join(VINDR_DIR, '**', '*'), recursive=True)):
        if os.path.splitext(path)[1].lower() in DICOM_EXTENSIONS:
            dicoms.setdefault(image_id(path), path)
    label_columns = [lookup[VINDR_COLUMNS[finding]] for finding in vindr_findings]
    grouped = labels.groupby(id_column)[label_columns].max()
    grouped = grouped[grouped.index.astype(str).isin(set(dicoms))]
    print(f"VinDr-CXR test images found on disk: {len(grouped)} of {labels[id_column].nunique()}.")
    vindr = pd.DataFrame({'key': grouped.index.astype(str)})
    vindr['patient'] = vindr['key']
    print("VinDr-CXR has no patient identifier, so each image is treated as its own patient.")
    vindr['path'] = vindr['key'].map(dicoms)
    for finding in vindr_findings:
        vindr[finding] = (grouped[lookup[VINDR_COLUMNS[finding]]].to_numpy(dtype=float) >= 0.5).astype(np.uint8)
    X_vindr = np.zeros((len(vindr), IMG_SIZE, IMG_SIZE, 1), dtype=np.uint8)
    sizes = np.zeros((len(vindr), 2), dtype=np.int64)
    for i, path in enumerate(vindr['path']):
        pixels = read_dicom(path)
        sizes[i] = pixels.shape[1], pixels.shape[0]
        X_vindr[i, :, :, 0] = np.asarray(Image.fromarray(pixels).resize((IMG_SIZE, IMG_SIZE), RESAMPLE), dtype=np.uint8)
        if i % 500 == 0:
            print(f"  Decoded {i}/{len(vindr)} DICOM files", flush=True)
    vindr['width'] = sizes[:, 0]
    vindr['height'] = sizes[:, 1]
    np.save('models/X_vindr.npy', X_vindr)
    np.save('models/Y_vindr.npy', vindr[vindr_findings].to_numpy(dtype=np.uint8))
    np.save('models/vindr_findings.npy', np.array(vindr_findings))
    vindr.to_csv('models/metadata_vindr.csv', index=False)
    print(f"VinDr-CXR: {len(vindr)} test radiographs saved at {IMG_SIZE}px.")
    print(vindr[vindr_findings].sum().to_string())

    boxes = pd.read_csv(find_file(VINDR_DIR, 'annotations_test.csv'))
    box_lookup = {column.strip().lower(): column for column in boxes.columns}
    finding_of = {name: finding for finding, name in VINDR_COLUMNS.items()}
    vindr_row_of = {key: i for i, key in enumerate(vindr['key'])}
    box_records = []
    for record in boxes.dropna(subset=[box_lookup['x_min']]).to_dict('records'):
        key = str(record[box_lookup['image_id']])
        finding = finding_of.get(str(record[box_lookup['class_name']]).strip().lower())
        if key in vindr_row_of and finding is not None:
            row = vindr_row_of[key]
            sx = IMG_SIZE / float(vindr.at[row, 'width'])
            sy = IMG_SIZE / float(vindr.at[row, 'height'])
            box_records.append({
                'row': row,
                'finding': finding,
                'x0': record[box_lookup['x_min']] * sx,
                'y0': record[box_lookup['y_min']] * sy,
                'x1': record[box_lookup['x_max']] * sx,
                'y1': record[box_lookup['y_max']] * sy,
            })
    vd_boxes = pd.DataFrame(box_records, columns=['row', 'finding', 'x0', 'y0', 'x1', 'y1'])
    vd_boxes.to_csv('models/gt_boxes_vindr.csv', index=False)
    print(f"VinDr-CXR: {len(vd_boxes)} consensus boxes on {vd_boxes['row'].nunique()} radiographs mapped to harmonised findings.")

    hybridgnet_vindr = 'models/hybridgnet_masks_vindr.csv'
    vindr_anatomy_ready = os.path.exists(hybridgnet_vindr)
    print(f"HybridGNet VinDr masks found: {vindr_anatomy_ready}")
    if vindr_anatomy_ready:
        vindr_keys = vindr['key'].tolist()
        vd_path = 'models/anatomy_vindr.npy'
        vd_quality, vd_shapes, vd_horizontal = build_anatomy(hybridgnet_vindr, vindr_keys, image_id, vd_path, 'Quality')
        np.save('models/anatomy_vindr_quality.npy', vd_quality)
        report_anatomy('VinDr-CXR', vd_quality, vd_horizontal, check='anatomical plausibility')
        found = np.isfinite(vd_shapes[:, 0])
        same_size = (vd_shapes[found] == vindr[['height', 'width']].to_numpy(dtype=float)[found]).all(axis=1)
        print(f"  Mask and image sizes agree for {int(same_size.sum())} of {int(found.sum())} radiographs.")
else:
    print(f"Label file not found in {VINDR_DIR}. Skipping the VinDr preparation.")

In [ ]:
def outline(ax, mask, colour):
    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    for contour in contours:
        closed = np.vstack([contour[:, 0, :], contour[:1, 0, :]])
        ax.plot(closed[:, 0], closed[:, 1], color=colour, linewidth=1)

def draw(ax, image, layers, title):
    ax.imshow(image, cmap='gray')
    outline(ax, layers & 1, 'cyan')
    outline(ax, (layers >> 1) & 1, 'lime')
    ax.set_title(title, fontsize=9)

def draw_boxes(ax, boxes):
    for box in boxes.itertuples():
        width, height = box.x1 - box.x0, box.y1 - box.y0
        ax.add_patch(plt.Rectangle((box.x0, box.y0), width, height, fill=False, edgecolor='yellow', linewidth=1.5))

In [ ]:
nih_images = np.load('models/X_test.npy', mmap_mode='r')
nih_anatomy = np.load('models/anatomy_nih.npy', mmap_mode='r')
cx_anatomy = np.load('models/anatomy_chexpert.npy', mmap_mode='r')
nih_pick = [row for row in nih_boxes['row'].unique() if nih_quality[row] >= MASK_QUALITY_MIN][:3]
cx_candidates = [row for row in np.unique(gt_rows) if cx_quality[row] >= MASK_QUALITY_MIN]
cx_val = [row for row in cx_candidates if chexpert.at[row, 'split'] == 'val']
cx_test = [row for row in cx_candidates if chexpert.at[row, 'split'] == 'test']
cx_pick = cx_val[:1] + cx_test[:2]
fig, axes = plt.subplots(3 if vindr_ready else 2, 3, figsize=(12, 12 if vindr_ready else 8), squeeze=False)
for ax, row in zip(axes[0], nih_pick):
    draw(ax, nih_images[row, :, :, 0], nih_anatomy[row], f"NIH {nih_keys[row]}")
    draw_boxes(ax, nih_boxes[nih_boxes['row'] == row])
for ax, row in zip(axes[1], cx_pick):
    draw(ax, X_chexpert[row, :, :, 0], cx_anatomy[row], f"CheXpert {chexpert.at[row, 'key']}")
    for mask in gt_masks[gt_rows == row]:
        outline(ax, mask, 'yellow')
if vindr_ready:
    vd_anatomy = np.load('models/anatomy_vindr.npy', mmap_mode='r')
    vd_pick = [row for row in vd_boxes['row'].unique() if vd_quality[row] >= MASK_QUALITY_MIN][:3]
    for ax, row in zip(axes[2], vd_pick):
        draw(ax, X_vindr[row, :, :, 0], vd_anatomy[row], f"VinDr-CXR {vindr.at[row, 'key'][:12]}")
        draw_boxes(ax, vd_boxes[vd_boxes['row'] == row])
for ax in axes.ravel():
    ax.axis('off')
plt.suptitle('Anatomy check: thoracic region (cyan), lungs (green), radiologist findings (yellow)')
plt.tight_layout()
plt.savefig('figures/anatomy_quality_check.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
def share_inside(mask, region):
    return float(np.logical_and(mask, region).sum() / mask.sum()) if mask.sum() else np.nan

def box_mask(box):
    mask = np.zeros((IMG_SIZE, IMG_SIZE), dtype=bool)
    top, bottom = int(np.floor(max(box.y0, 0))), int(np.ceil(min(box.y1, IMG_SIZE)))
    left, right = int(np.floor(max(box.x0, 0))), int(np.ceil(min(box.x1, IMG_SIZE)))
    mask[top:bottom, left:right] = True
    return mask

In [ ]:
coverage = []
box_sets = [('NIH', nih_boxes, nih_quality, nih_anatomy)]
if vindr_ready:
    box_sets.append(('VinDr-CXR', vd_boxes, vd_quality, vd_anatomy))
for name, boxes, quality, anatomy in box_sets:
    for box in boxes.itertuples():
        if quality[box.row] >= MASK_QUALITY_MIN:
            region = (anatomy[box.row] & 1).astype(bool)
            coverage.append({'dataset': name, 'finding': box.finding, 'inside': share_inside(box_mask(box), region)})
            
for row, finding, mask in zip(gt_rows, gt_findings, gt_masks):
    if cx_quality[row] >= MASK_QUALITY_MIN:
        region = (cx_anatomy[row] & 1).astype(bool)
        coverage.append({'dataset': 'CheXpert', 'finding': finding, 'inside': share_inside(mask, region)})
coverage = pd.DataFrame(coverage).dropna()
coverage.to_csv('models/region_coverage.csv', index=False)
for name, part in coverage.groupby('dataset'):
    print(f"{name}: {len(part)} annotations compared with the thoracic region.")
    inside = part['inside']
    print(f"  Mean share inside: {100 * inside.mean():.1f}%")
    print(f"  Median share inside: {100 * inside.median():.1f}%")
    print(f"  5th percentile: {100 * inside.quantile(0.05):.1f}%")
print(coverage.groupby(['dataset', 'finding'])['inside'].mean().mul(100).round(1).to_string())